# LeetCode 2100 — Find Good Days to Rob the Bank bằng Python

Link đề: https://leetcode.com/problems/find-good-days-to-rob-the-bank/ · Độ khó: **Medium** · Chủ đề: Array, Dynamic Programming, Prefix Sum

Skeleton LeetCode:

```python
class Solution:
    def goodDaysToRobBank(self, security: List[int], time: int) -> List[int]:
        ...
```

Lộ trình:
1. **Brute force**: với mỗi ngày, kiểm tra `time` ngày trước không tăng và `time` ngày sau không giảm — `O(n · time)`.
2. **Độ dài chuỗi đơn điệu (DP hai mảng)**: `left[i]` = số bước "không tăng" liên tiếp kết thúc ở `i`, `right[i]` = số bước "không giảm" liên tiếp bắt đầu ở `i` — `O(n)` (**nộp bài**).
3. **Prefix đếm vi phạm**: đếm số bước "tăng" trong đoạn trước và số bước "giảm" trong đoạn sau bằng prefix sum — `O(n)`.
4. **Cửa sổ trượt đếm vi phạm**: cùng ý tưởng Cách 3 nhưng giữ bộ đếm khi trượt — `O(n)`, bộ nhớ `O(1)` ngoài đầu vào.

## 1. Mục tiêu học

1. Dịch điều kiện "chuỗi `security[i−time] ≥ … ≥ security[i] ≤ … ≤ security[i+time]`" thành hai điều kiện về **bước** giữa các ngày kề nhau.
2. Dùng **DP độ dài chuỗi liên tiếp**: `left[i] = left[i−1] + 1` nếu không tăng, ngược lại `0` — mẫu chung cho mọi bài "chuỗi đơn điệu liên tiếp".
3. Dùng **prefix sum trên mảng chỉ báo** để hỏi "đoạn này có vi phạm nào không?" trong `O(1)`.
4. Phân biệt cẩn thận **không tăng** (`≥`) với **giảm ngặt** (`>`), và "số bước" với "số phần tử".
5. Kiểm thử biên: `time = 0`, `2·time ≥ n`, mảng hằng.

## 2. Đề bài & ví dụ

`security[i]` là số bảo vệ ngày `i`. Ngày `i` là **ngày tốt** nếu:
- có ít nhất `time` ngày trước và `time` ngày sau `i`;
- số bảo vệ trong `time` ngày trước `i` (tính tới `i`) **không tăng**: `security[i−time] ≥ … ≥ security[i]`;
- số bảo vệ từ `i` tới `time` ngày sau **không giảm**: `security[i] ≤ … ≤ security[i+time]`.

Trả về mọi ngày tốt (thứ tự tùy ý; ở đây trả theo thứ tự tăng dần).

**Ví dụ 1:** `security = [5,3,3,3,5,6,2]`, `time = 2` → `[2,3]`.

**Ví dụ 2:** `security = [1,1,1,1,1]`, `time = 0` → `[0,1,2,3,4]`.

**Ví dụ 3:** `security = [1,2,3,4,5,6]`, `time = 2` → `[]`.

**Ràng buộc:** `1 ≤ n ≤ 10⁵`, `0 ≤ security[i], time ≤ 10⁵`.

In [1]:
import random
import time as _time
from itertools import accumulate
from typing import List

examples = [(([5, 3, 3, 3, 5, 6, 2], 2), [2, 3]), (([1, 1, 1, 1, 1], 0), [0, 1, 2, 3, 4]), (([1, 2, 3, 4, 5, 6], 2), [])]
for (sec, t), want in examples:
    print(f"security = {sec}, time = {t} -> {want}")

security = [5, 3, 3, 3, 5, 6, 2], time = 2 -> [2, 3]
security = [1, 1, 1, 1, 1], time = 0 -> [0, 1, 2, 3, 4]
security = [1, 2, 3, 4, 5, 6], time = 2 -> []


(Lưu ý: tham số tên `time` trùng tên module chuẩn `time`; notebook import module dưới tên `_time` để tránh nhầm.)

## 3. Kiến thức nền

### 3.1 Chuỗi không tăng ⇔ mọi bước kề không tăng

`a[l] ≥ a[l+1] ≥ … ≥ a[r]` ⇔ với mọi `j ∈ (l, r]`: `a[j] ≤ a[j−1]`. Điều kiện trên đoạn quy về điều kiện trên **từng bước**.

In [2]:
sec = [5, 3, 3, 3, 5, 6, 2]
steps = ["≤" if sec[j] <= sec[j - 1] else ">" for j in range(1, len(sec))]
print("bước j (so với j-1):", list(zip(range(1, len(sec)), steps)))

bước j (so với j-1): [(1, '≤'), (2, '≤'), (3, '≤'), (4, '>'), (5, '>'), (6, '≤')]


### 3.2 DP độ dài chuỗi liên tiếp

`left[i]` = số bước "không tăng" liên tiếp kết thúc tại `i` = `left[i−1] + 1` nếu `a[i] ≤ a[i−1]`, ngược lại `0`.

In [3]:
left = [0] * len(sec)
for i in range(1, len(sec)):
    left[i] = left[i - 1] + 1 if sec[i] <= sec[i - 1] else 0
print("left =", left)

left = [0, 1, 2, 3, 0, 0, 1]


### 3.3 Prefix sum trên mảng chỉ báo

`up[j] = 1` nếu `a[j] > a[j−1]` (vi phạm "không tăng"). Số vi phạm trong các bước `(l, r]` = `P[r] − P[l]` với `P` là prefix của `up`.

In [4]:
up = [0] + [1 if sec[j] > sec[j - 1] else 0 for j in range(1, len(sec))]
P = list(accumulate(up))
print("up =", up, "| P =", P, "| số bước tăng trong (0, 3] =", P[3] - P[0])

up = [0, 0, 0, 0, 1, 1, 0] | P = [0, 0, 0, 0, 1, 2, 2] | số bước tăng trong (0, 3] = 0


## 4. Ý tưởng nền tảng / trực giác

Ví dụ 1, `time = 2`. Tính `left[i]` (bước không tăng liên tiếp kết thúc ở `i`) và `right[i]` (bước không giảm liên tiếp bắt đầu ở `i`):

| `i` | 0 | 1 | 2 | 3 | 4 | 5 | 6 |
|---|---|---|---|---|---|---|---|
| `security` | 5 | 3 | 3 | 3 | 5 | 6 | 2 |
| `left` | 0 | 1 | 2 | 3 | 0 | 0 | 1 |
| `right` | 0 | 4 | 3 | 2 | 1 | 0 | 0 |

Tính `right` từ phải sang: `right[6] = 0`; `right[5]`: `6 ≤ 2`? không ⇒ `0`; `right[4]`: `5 ≤ 6` ⇒ `1`; `right[3]`: `3 ≤ 5` ⇒ `2`; `right[2]`: `3 ≤ 3` ⇒ `3`; `right[1]`: `3 ≤ 3` ⇒ `4`; `right[0]`: `5 ≤ 3`? không ⇒ `0`.

Ngày tốt ⇔ `left[i] ≥ time` **và** `right[i] ≥ time`: `i = 2` (`2, 3`) ✓, `i = 3` (`3, 2`) ✓, `i = 1` (`1, 4`) ✗. ⇒ `[2, 3]` ✓.

In [5]:
right = [0] * len(sec)
for i in range(len(sec) - 2, -1, -1):
    right[i] = right[i + 1] + 1 if sec[i] <= sec[i + 1] else 0
print("left  =", left)
print("right =", right)
print("ngày tốt (time = 2):", [i for i in range(len(sec)) if left[i] >= 2 and right[i] >= 2])

left  = [0, 1, 2, 3, 0, 0, 1]
right = [0, 4, 3, 2, 1, 0, 0]
ngày tốt (time = 2): [2, 3]


## 5. Cách 1 — Brute Force / baseline: kiểm tra từng ngày

In [6]:
def good_days_brute(security, time):
    n = len(security); res = []
    for i in range(time, n - time):
        ok = all(security[j] <= security[j - 1] for j in range(i - time + 1, i + 1)) and \
             all(security[j] <= security[j + 1] for j in range(i, i + time))
        if ok:
            res.append(i)
    return res

for (sec, t), want in examples:
    print(good_days_brute(sec, t), "kỳ vọng", want)

[2, 3] kỳ vọng [2, 3]
[0, 1, 2, 3, 4] kỳ vọng [0, 1, 2, 3, 4]
[] kỳ vọng []


**Độ phức tạp:** `O((n − 2·time) · 2·time)` — tệ nhất khi `time ≈ n/4`: `≈ n²/4 = 2.5·10⁹` với `n = 10⁵` (ví dụ mảng hằng, mọi phép kiểm đều chạy hết). Space `O(1)`.

**Nút thắt:** các đoạn của hai ngày kề nhau chồng nhau gần hết, nhưng ta kiểm tra lại từ đầu.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight:** "đoạn `time` bước trước `i` không tăng" ⇔ **chuỗi không tăng liên tiếp kết thúc ở `i` dài ít nhất `time` bước** ⇔ `left[i] ≥ time`. Tương tự `right[i] ≥ time`. Cả `left` và `right` tính một lượt bằng DP.

**Hai định nghĩa tương đương (Cách 3):** ngày `i` tốt ⇔ trong các bước `(i − time, i]` không có bước **tăng** và trong các bước `(i, i + time]` không có bước **giảm** ⇔ hai hiệu prefix bằng `0`.

**Quy tắc:**
- *Nếu* `a[i] ≤ a[i−1]` *thì* `left[i] = left[i−1] + 1`, *ngược lại* `0`.
- *Nếu* `a[i] ≤ a[i+1]` *thì* `right[i] = right[i+1] + 1`, *ngược lại* `0`.
- Điều kiện "có đủ `time` ngày hai bên" được ngầm bảo đảm: `left[i] ≤ i`, `right[i] ≤ n − 1 − i`.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Hai mảng độ dài chuỗi (code nộp)

In [7]:
def good_days_runs(security, time):
    n = len(security)
    left = [0] * n; right = [0] * n
    for i in range(1, n):
        if security[i] <= security[i - 1]:
            left[i] = left[i - 1] + 1
    for i in range(n - 2, -1, -1):
        if security[i] <= security[i + 1]:
            right[i] = right[i + 1] + 1
    return [i for i in range(n) if left[i] >= time and right[i] >= time]

for (sec, t), want in examples:
    print(good_days_runs(sec, t), "kỳ vọng", want)

[2, 3] kỳ vọng [2, 3]
[0, 1, 2, 3, 4] kỳ vọng [0, 1, 2, 3, 4]
[] kỳ vọng []


Code nộp LeetCode của Cách 2: xem mục 9.

**Độ phức tạp:** `O(n)` thời gian, `O(n)` bộ nhớ.

### 7.2 Cách 3 — Prefix đếm vi phạm

In [8]:
def good_days_prefix(security, time):
    n = len(security)
    up = [0] * n; down = [0] * n             # up[j] = 1 nếu a[j] > a[j-1]; down[j] = 1 nếu a[j] < a[j-1]
    for j in range(1, n):
        up[j] = 1 if security[j] > security[j - 1] else 0
        down[j] = 1 if security[j] < security[j - 1] else 0
    PU = list(accumulate(up)); PD = list(accumulate(down))
    res = []
    for i in range(time, n - time):
        if PU[i] - PU[i - time] == 0 and PD[i + time] - PD[i] == 0:
            res.append(i)
    return res

for (sec, t), want in examples:
    print(good_days_prefix(sec, t), "kỳ vọng", want)

[2, 3] kỳ vọng [2, 3]
[0, 1, 2, 3, 4] kỳ vọng [0, 1, 2, 3, 4]
[] kỳ vọng []


Các bước trước `i` là `j = i − time + 1 .. i` ⇒ số bước tăng `= PU[i] − PU[i − time]`. Các bước sau là `j = i + 1 .. i + time` ⇒ số bước giảm `= PD[i + time] − PD[i]`.

**Độ phức tạp:** `O(n)` thời gian, `O(n)` bộ nhớ. Ưu điểm: mở rộng được cho "cho phép tối đa `c` vi phạm" (Bài 5) — chỉ đổi `== 0` thành `≤ c`.

### 7.3 Cách 4 — Cửa sổ trượt đếm vi phạm

Duyệt `i` từ `time` tới `n − 1 − time`; giữ `bad_before` = số bước tăng trong `(i − time, i]` và `bad_after` = số bước giảm trong `(i, i + time]`. Khi `i` tăng 1, mỗi cửa sổ thêm một bước và bỏ một bước.

In [9]:
def good_days_window(security, time):
    n = len(security); res = []
    if 2 * time >= n:
        return res
    is_up = lambda j: security[j] > security[j - 1]      # bước j (1 <= j < n)
    is_down = lambda j: security[j] < security[j - 1]
    i = time
    bad_before = sum(is_up(j) for j in range(1, i + 1))  # bước 1..time
    bad_after = sum(is_down(j) for j in range(i + 1, i + time + 1))
    while True:
        if bad_before == 0 and bad_after == 0:
            res.append(i)
        if i + time + 1 >= n:
            break
        # trượt sang i + 1
        if time > 0:
            bad_before += is_up(i + 1) - is_up(i - time + 1)
            bad_after += is_down(i + time + 1) - is_down(i + 1)
        i += 1
    return res

for (sec, t), want in examples:
    print(good_days_window(sec, t), "kỳ vọng", want)

[2, 3] kỳ vọng [2, 3]
[0, 1, 2, 3, 4] kỳ vọng [0, 1, 2, 3, 4]
[] kỳ vọng []


**Độ phức tạp:** `O(n)` thời gian, `O(1)` bộ nhớ phụ (ngoài kết quả). Khi `time = 0` cả hai cửa sổ rỗng ⇒ mọi ngày đều tốt.

## 8. TRACE từng bước

Trace Cách 2 trên ví dụ 1: in `left`, `right` qua từng bước và quyết định cho mỗi ngày.

In [10]:
def trace_runs(security, time):
    n = len(security); left = [0] * n; right = [0] * n
    print("Tính left (trái -> phải):")
    for i in range(1, n):
        cond = security[i] <= security[i - 1]
        left[i] = left[i - 1] + 1 if cond else 0
        print(f"  i = {i}: {security[i]} <= {security[i - 1]}? {cond} -> left[{i}] = {left[i]}")
    print("Tính right (phải -> trái):")
    for i in range(n - 2, -1, -1):
        cond = security[i] <= security[i + 1]
        right[i] = right[i + 1] + 1 if cond else 0
        print(f"  i = {i}: {security[i]} <= {security[i + 1]}? {cond} -> right[{i}] = {right[i]}")
    for i in range(n):
        print(f"  ngày {i}: left = {left[i]}, right = {right[i]} -> {'TỐT' if left[i] >= time and right[i] >= time else '-'}")

trace_runs([5, 3, 3, 3, 5, 6, 2], 2)

Tính left (trái -> phải):
  i = 1: 3 <= 5? True -> left[1] = 1
  i = 2: 3 <= 3? True -> left[2] = 2
  i = 3: 3 <= 3? True -> left[3] = 3
  i = 4: 5 <= 3? False -> left[4] = 0
  i = 5: 6 <= 5? False -> left[5] = 0
  i = 6: 2 <= 6? True -> left[6] = 1
Tính right (phải -> trái):
  i = 5: 6 <= 2? False -> right[5] = 0
  i = 4: 5 <= 6? True -> right[4] = 1
  i = 3: 3 <= 5? True -> right[3] = 2
  i = 2: 3 <= 3? True -> right[2] = 3
  i = 1: 3 <= 3? True -> right[1] = 4
  i = 0: 5 <= 3? False -> right[0] = 0
  ngày 0: left = 0, right = 0 -> -
  ngày 1: left = 1, right = 4 -> -
  ngày 2: left = 2, right = 3 -> TỐT
  ngày 3: left = 3, right = 2 -> TỐT
  ngày 4: left = 0, right = 1 -> -
  ngày 5: left = 0, right = 0 -> -
  ngày 6: left = 1, right = 0 -> -


In [11]:
def trace_window(security, time):
    n = len(security)
    up = [0] + [int(security[j] > security[j - 1]) for j in range(1, n)]
    dn = [0] + [int(security[j] < security[j - 1]) for j in range(1, n)]
    print("up   =", up); print("down =", dn)
    for i in range(time, n - time):
        bb = sum(up[i - time + 1:i + 1]); ba = sum(dn[i + 1:i + time + 1])
        print(f"  ngày {i}: bước trước {list(range(i - time + 1, i + 1))} có {bb} lần tăng; "
              f"bước sau {list(range(i + 1, i + time + 1))} có {ba} lần giảm -> {'TỐT' if bb == ba == 0 else '-'}")

trace_window([5, 3, 3, 3, 5, 6, 2], 2)

up   = [0, 0, 0, 0, 1, 1, 0]
down = [0, 1, 0, 0, 0, 0, 1]
  ngày 2: bước trước [1, 2] có 0 lần tăng; bước sau [3, 4] có 0 lần giảm -> TỐT
  ngày 3: bước trước [2, 3] có 0 lần tăng; bước sau [4, 5] có 0 lần giảm -> TỐT
  ngày 4: bước trước [3, 4] có 1 lần tăng; bước sau [5, 6] có 1 lần giảm -> -


## 9. Code nộp LeetCode

In [12]:
class Solution:
    def goodDaysToRobBank(self, security: List[int], time: int) -> List[int]:
        n = len(security)
        left = [0] * n
        right = [0] * n
        for i in range(1, n):
            if security[i] <= security[i - 1]:        # không tăng (cho phép bằng)
                left[i] = left[i - 1] + 1
        for i in range(n - 2, -1, -1):
            if security[i] <= security[i + 1]:        # không giảm (cho phép bằng)
                right[i] = right[i + 1] + 1
        return [i for i in range(n) if left[i] >= time and right[i] >= time]

sol = Solution()
for (sec, t), want in examples:
    got = sol.goodDaysToRobBank(sec, t)
    print(got, "OK" if sorted(got) == sorted(want) else "SAI")

[2, 3] OK
[0, 1, 2, 3, 4] OK
[] OK


## 10. Vì sao thuật toán đúng

**Bổ đề 1.** `left[i]` = số `L` lớn nhất sao cho `a[i−L] ≥ a[i−L+1] ≥ … ≥ a[i]`.

*Quy nạp:* `left[0] = 0` đúng. Nếu `a[i] ≤ a[i−1]` thì chuỗi không tăng dài nhất kết thúc ở `i` là chuỗi dài nhất kết thúc ở `i−1` nối thêm bước `i−1 → i` ⇒ `left[i−1] + 1`. Nếu `a[i] > a[i−1]` thì không chuỗi không tăng nào kết thúc ở `i` có thể chứa bước này ⇒ `0`. ✓

**Bổ đề 2.** Tương tự, `right[i]` = số `R` lớn nhất sao cho `a[i] ≤ … ≤ a[i+R]`.

**Hệ quả.** Ngày `i` tốt ⇔ `time` bước trước `i` không tăng ⇔ `left[i] ≥ time` (vì chuỗi dài nhất chứa mọi chuỗi ngắn hơn kết thúc ở `i`); tương tự phía sau. Điều kiện "có đủ `time` ngày hai bên" tự thỏa vì `left[i] ≤ i` và `right[i] ≤ n − 1 − i`. ✓

**Cách 3/4.** Đoạn bước không chứa bước tăng ⇔ đoạn không tăng (mục 3.1); prefix/cửa sổ đếm đúng số bước tăng/giảm trong đoạn. ✓

## 11. Phân tích Time / Space

`n ≤ 10⁵`.

| Cách | Time | Space phụ |
|---|---|---|
| 1. Brute force | `O(n · time)` | `O(1)` |
| 2. Hai mảng độ dài | `O(n)` | `O(n)` |
| 3. Prefix vi phạm | `O(n)` | `O(n)` |
| 4. Cửa sổ trượt | `O(n)` | `O(1)` |

In [13]:
rng = random.Random(1)
const = [7] * 10**5                                   # mảng hằng: brute force chạy lâu nhất
rand = [rng.randint(0, 3) for _ in range(10**5)]
valley = list(range(50000, 0, -1)) + list(range(0, 50000))
for name, arr, t in [("hằng, time = 1000", const, 1000), ("ngẫu nhiên 0..3, time = 2", rand, 2), ("thung lũng, time = 40000", valley, 40000)]:
    res = []
    for fname, fn in [("runs", good_days_runs), ("prefix", good_days_prefix), ("window", good_days_window), ("submit", Solution().goodDaysToRobBank)]:
        t0 = _time.perf_counter(); r = fn(arr, t); res.append(r)
        print(f"{name:26s} {fname:7s}: {len(r):>6} ngày tốt  {(_time.perf_counter() - t0) * 1000:.1f} ms")
    assert all(x == res[0] for x in res)
t0 = _time.perf_counter(); good_days_brute(const[:20000], 1000); print(f"brute (n = 20000, time = 1000): {_time.perf_counter() - t0:.2f}s")

hằng, time = 1000          runs   :  98000 ngày tốt  24.8 ms
hằng, time = 1000          prefix :  98000 ngày tốt  16.5 ms
hằng, time = 1000          window :  98000 ngày tốt  29.6 ms
hằng, time = 1000          submit :  98000 ngày tốt  23.2 ms


ngẫu nhiên 0..3, time = 2  runs   :  14202 ngày tốt  16.6 ms


ngẫu nhiên 0..3, time = 2  prefix :  14202 ngày tốt  25.0 ms


ngẫu nhiên 0..3, time = 2  window :  14202 ngày tốt  33.8 ms
ngẫu nhiên 0..3, time = 2  submit :  14202 ngày tốt  13.5 ms
thung lũng, time = 40000   runs   :      1 ngày tốt  12.1 ms
thung lũng, time = 40000   prefix :      1 ngày tốt  12.3 ms
thung lũng, time = 40000   window :      1 ngày tốt  10.2 ms
thung lũng, time = 40000   submit :      1 ngày tốt  11.9 ms


brute (n = 20000, time = 1000): 1.42s


## 12. Bộ test

In [14]:
fns = [good_days_runs, good_days_prefix, good_days_window, Solution().goodDaysToRobBank]

for (sec, t), want in examples:
    assert good_days_brute(sec, t) == want
    for fn in fns:
        assert sorted(fn(sec, t)) == want

edge = [
    (([1], 0), [0]), (([1], 1), []),                          # n = 1
    (([3, 2, 1], 1), []), (([1, 2, 3], 1), []), (([3, 1, 3], 1), [1]),
    (([2, 2, 2], 1), [1]),                                    # bằng nhau được phép
    (([4, 3, 2, 1, 2, 3, 4], 3), [3]),                        # đúng 2*time + 1 = n
    (([4, 3, 2, 1, 2, 3, 4], 4), []),                         # 2*time >= n
    (([1, 1, 1, 1, 1], 2), [2]),
]
for (sec, t), want in edge:
    assert good_days_brute(sec, t) == want, (sec, t)
    for fn in fns:
        assert sorted(fn(sec, t)) == want, (fn, sec, t)

rng = random.Random(2024)
for _ in range(2000):
    n = rng.randint(1, 25)
    sec = [rng.randint(0, rng.choice([1, 2, 5, 100])) for _ in range(n)]
    t = rng.randint(0, n)
    want = good_days_brute(sec, t)
    for fn in fns:
        assert sorted(fn(sec, t)) == want, (fn, sec, t)
print("ví dụ + edge + 2000 bộ ngẫu nhiên khớp brute force")

ví dụ + edge + 2000 bộ ngẫu nhiên khớp brute force


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Dùng so sánh **ngặt** (`<`) thay vì `≤`

In [15]:
def wrong_strict(security, time):
    n = len(security); left = [0] * n; right = [0] * n
    for i in range(1, n):
        if security[i] < security[i - 1]: left[i] = left[i - 1] + 1        # SAI: đề cho phép bằng
    for i in range(n - 2, -1, -1):
        if security[i] < security[i + 1]: right[i] = right[i + 1] + 1
    return [i for i in range(n) if left[i] >= time and right[i] >= time]

for (sec, t), want in examples:
    print("sai =", wrong_strict(sec, t), "| đúng =", want)

sai = [] | đúng = [2, 3]
sai = [0, 1, 2, 3, 4] | đúng = [0, 1, 2, 3, 4]
sai = [] | đúng = []


"Non-increasing" là **không tăng** (cho phép bằng). Ví dụ 1 có các ngày `3, 3, 3` liên tiếp — so sánh ngặt loại mất. Lưu ý ví dụ 2 vẫn đúng vì `time = 0`. **Sửa:** `≤`.

### Lỗi 2 — Nhầm hướng so sánh của `right` (tính từ trái sang)

In [16]:
def wrong_right_forward(security, time):
    n = len(security); left = [0] * n; right = [0] * n
    for i in range(1, n):
        if security[i] <= security[i - 1]: left[i] = left[i - 1] + 1
        if security[i] >= security[i - 1]: right[i] = right[i - 1] + 1      # SAI: chuỗi không giảm KẾT THÚC ở i
    return [i for i in range(n) if left[i] >= time and right[i] >= time]

for (sec, t), want in examples + [(([3, 1, 3], 1), [1])]:
    print("sai =", wrong_right_forward(sec, t), "| đúng =", want)

sai = [3] | đúng = [2, 3]
sai = [0, 1, 2, 3, 4] | đúng = [0, 1, 2, 3, 4]
sai = [] | đúng = []
sai = [] | đúng = [1]


`right[i]` phải đo chuỗi không giảm **bắt đầu** tại `i` (nhìn về tương lai), nên phải tính **từ phải sang trái**. Tính xuôi cho chuỗi không giảm **kết thúc** tại `i` — thông tin về quá khứ. **Sửa:** vòng lặp ngược.

### Lỗi 3 — Lẫn "số bước" với "số phần tử" (`≥ time + 1` hoặc `> time`)

In [17]:
def wrong_count_elements(security, time):
    n = len(security); left = [1] * n; right = [1] * n                     # đếm số PHẦN TỬ trong chuỗi
    for i in range(1, n):
        if security[i] <= security[i - 1]: left[i] = left[i - 1] + 1
    for i in range(n - 2, -1, -1):
        if security[i] <= security[i + 1]: right[i] = right[i + 1] + 1
    return [i for i in range(n) if left[i] >= time and right[i] >= time]  # SAI: cần >= time + 1 phần tử

for (sec, t), want in examples + [(([2, 1, 2], 2), [])]:
    print("sai =", wrong_count_elements(sec, t), "| đúng =", want)

sai = [1, 2, 3] | đúng = [2, 3]
sai = [0, 1, 2, 3, 4] | đúng = [0, 1, 2, 3, 4]
sai = [] | đúng = []
sai = [1] | đúng = []


`time` ngày trước **cộng** ngày `i` là `time + 1` phần tử, tức `time` bước. Đếm phần tử thì phải so với `time + 1`. **Sửa:** thống nhất một quy ước (notebook dùng "số bước").

### Lỗi 4 — Cửa sổ prefix lệch một bước

In [18]:
def wrong_prefix_shift(security, time):
    n = len(security)
    up = [0] + [1 if security[j] > security[j - 1] else 0 for j in range(1, n)]
    dn = [0] + [1 if security[j] < security[j - 1] else 0 for j in range(1, n)]
    PU = list(accumulate(up)); PD = list(accumulate(dn)); res = []
    for i in range(time, n - time):
        if PU[i] - PU[i - time] == 0 and PD[i + time - 1] - PD[i - 1 if i > 0 else 0] == 0:   # SAI: lệch đoạn sau
            res.append(i)
    return res

rng = random.Random(4); shown = 0
for _ in range(5000):
    sec = [rng.randint(0, 3) for _ in range(rng.randint(2, 8))]; t = rng.randint(1, 3)
    if wrong_prefix_shift(sec, t) != good_days_brute(sec, t):
        print(f"phản ví dụ: {sec}, time = {t} | sai = {wrong_prefix_shift(sec, t)} | đúng = {good_days_brute(sec, t)}")
        shown += 1
        if shown == 2: break

phản ví dụ: [2, 0, 1, 2, 2, 1], time = 1 | sai = [4] | đúng = [1]
phản ví dụ: [1, 0, 2, 2], time = 1 | sai = [] | đúng = [1]


Bước sau ngày `i` là `j = i + 1 .. i + time` ⇒ `PD[i + time] − PD[i]`. Viết `PD[i + time − 1] − PD[i − 1]` vừa bỏ sót bước cuối vừa tính thừa bước `i` (bước **vào** ngày `i`, thuộc phía trước). **Sửa:** vẽ rõ chỉ số bước trước khi viết prefix.

### Lỗi 5 — Quên `time = 0` / `2·time ≥ n` trong cài đặt cửa sổ

Cửa sổ trượt khởi tạo bằng `range(1, time + 1)`; nếu không chặn `2·time ≥ n` thì `i + time` vượt mảng. Cách 2 không có vấn đề này (điều kiện ngầm trong `left`, `right`). Test `([1], 1)` và `([1], 0)`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Đoạn `k` phần tử trước/sau `i` đơn điệu" ⇒ **độ dài chuỗi đơn điệu liên tiếp** từ hai phía (`left`, `right`).
- Mọi điều kiện "cả đoạn thỏa P" mà P kiểm tra theo bước kề ⇒ đếm **vi phạm** bằng prefix sum.
- Luôn phân biệt `≥` (không tăng) và `>` (giảm ngặt).
- Họ bài: 845 *Longest Mountain in Array*, 1671 *Minimum Number of Removals to Make Mountain Array*, 2420 *Find All Good Indices* (gần như cùng bài, loại trừ ngày `i`), 1800 *Maximum Ascending Subarray Sum*.

## 15. Biến thể của bài

1. **2420 Find All Good Indices:** `k` phần tử **ngay trước** `i` không tăng và `k` phần tử **ngay sau** `i` không giảm — **không** tính chính `i`. Dịch chỉ số `left[i−1] ≥ k − 1`, `right[i+1] ≥ k − 1`.
2. **Ngày "xấu nhất"** (đỉnh núi): đổi chiều so sánh.
3. **Cho phép tối đa `c` vi phạm** mỗi phía (Bài 5): prefix vi phạm.
4. **Thung lũng dài nhất** (845 cho núi): `max(left[i] + right[i])` với `left[i], right[i] > 0`.

In [19]:
def good_indices_2420(nums, k):
    """Biến thể 1: k phần tử trước i (không gồm i) không tăng, k phần tử sau i (không gồm i) không giảm."""
    n = len(nums); dec = [1] * n; inc = [1] * n
    for i in range(1, n):
        if nums[i] <= nums[i - 1]: dec[i] = dec[i - 1] + 1
    for i in range(n - 2, -1, -1):
        if nums[i] <= nums[i + 1]: inc[i] = inc[i + 1] + 1
    return [i for i in range(k, n - k) if dec[i - 1] >= k and inc[i + 1] >= k]

def good_indices_brute(nums, k):
    n = len(nums)
    return [i for i in range(k, n - k)
            if all(nums[j] >= nums[j + 1] for j in range(i - k, i - 1)) and all(nums[j] <= nums[j + 1] for j in range(i + 1, i + k))]

def longest_valley(a):
    """Biến thể 4: độ dài (số phần tử) thung lũng dài nhất có cả hai sườn (giảm ngặt rồi tăng ngặt)."""
    n = len(a); L = [0] * n; R = [0] * n
    for i in range(1, n):
        if a[i] < a[i - 1]: L[i] = L[i - 1] + 1
    for i in range(n - 2, -1, -1):
        if a[i] < a[i + 1]: R[i] = R[i + 1] + 1
    return max((L[i] + R[i] + 1 for i in range(n) if L[i] and R[i]), default=0)

rng = random.Random(15)
for _ in range(1000):
    nums = [rng.randint(0, 4) for _ in range(rng.randint(1, 15))]; k = rng.randint(1, 5)
    assert good_indices_2420(nums, k) == good_indices_brute(nums, k)
print("2420 ví dụ [2,1,1,1,3,4,1], k = 2 ->", good_indices_2420([2, 1, 1, 1, 3, 4, 1], 2), "(kỳ vọng [2, 3])")
print("thung lũng dài nhất [5,3,1,2,6,4,3,5]:", longest_valley([5, 3, 1, 2, 6, 4, 3, 5]))

2420 ví dụ [2,1,1,1,3,4,1], k = 2 -> [2, 3] (kỳ vọng [2, 3])
thung lũng dài nhất [5,3,1,2,6,4,3,5]: 5


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space phụ | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute force | kiểm tra từng ngày | `O(n · time)` | `O(1)` | kiểm chứng |
| 2. Hai mảng độ dài | DP chuỗi liên tiếp | `O(n)` | `O(n)` | **nộp bài** |
| 3. Prefix vi phạm | đếm bước sai | `O(n)` | `O(n)` | cho phép `c` vi phạm |
| 4. Cửa sổ trượt | bộ đếm trượt | `O(n)` | `O(1)` | bộ nhớ hạn chế |

## 17. Bài tập (từ dễ đến khó)

**Bài 1 (đoán output):** `[1, 0, 1]`, `time = 1`? `[3, 3, 3, 3]`, `time = 1`? `[5, 4, 3, 2, 1]`, `time = 1`?

**Bài 2 (tính tay):** `security = [6, 5, 5, 4, 4, 5, 7, 7, 2]`, `time = 2`. Tính `left`, `right` và các ngày tốt.

**Bài 3 (điền TODO):** hoàn thiện `nonincreasing_runs(a)` trả về mảng `left` (số bước không tăng liên tiếp kết thúc ở mỗi vị trí).

**Bài 4 (tự viết lại):** viết `max_time(security)` = giá trị `time` **lớn nhất** sao cho tồn tại ít nhất một ngày tốt.

**Bài 5 (mini challenge):** "ngày tốt với dung sai `c`": mỗi phía được phép có tối đa `c` bước vi phạm (bước tăng ở phía trước, bước giảm ở phía sau). Trả về danh sách ngày tốt trong `O(n)`. Kiểm chứng bằng brute force. (Biến thể tự đặt — đối chiếu lại đề gốc nếu dùng làm đề.)

In [20]:
def nonincreasing_runs(a):
    # TODO: left[0] = 0; left[i] = left[i-1] + 1 nếu a[i] <= a[i-1], ngược lại 0
    pass

## 18. Gợi ý 3 mức

**Bài 1:** (1) `[1,0,1]`: ngày 1 có `1 ≥ 0 ≤ 1`. (2) `[3,3,3,3]`, `time = 1`: ngày 1, 2. (3) `[5,4,3,2,1]`: phía sau phải không giảm — không ngày nào ⇒ `[]`.

**Bài 2:** (1) `left = [0,1,2,3,4,0,0,1,2]`. (2) Tính `right` từ phải: `right[8] = 0`; `7 ≤ 2`? không ⇒ `right[7] = 0`; `7 ≤ 7` ⇒ `right[6] = 1`; `5 ≤ 7` ⇒ `right[5] = 2`; `4 ≤ 5` ⇒ `right[4] = 3`; `4 ≤ 4` ⇒ `right[3] = 4`; `5 ≤ 4`? không ⇒ `right[2] = 0`; `5 ≤ 5` ⇒ `right[1] = 1`; `6 ≤ 5`? không ⇒ `right[0] = 0`. Vậy `right = [0,1,0,4,3,2,1,0,0]`. (3) Ngày tốt: `left ≥ 2` và `right ≥ 2` ⇒ `[3, 4]`.

**Bài 3:** (1) Mảng `n` số 0. (2) Một vòng `for` từ 1. (3) So sánh `≤`.

**Bài 4:** (1) Ngày `i` tốt với mọi `time ≤ min(left[i], right[i])`. (2) Đáp án `max_i min(left[i], right[i])`. (3) Luôn `≥ 0` (với `time = 0` mọi ngày tốt).

**Bài 5:** (1) Dùng Cách 3. (2) Điều kiện `PU[i] − PU[i − time] ≤ c` và `PD[i + time] − PD[i] ≤ c`. (3) Brute force: đếm trực tiếp số bước vi phạm.

## 19. Đáp án tham khảo

In [21]:
# Bài 1
assert [Solution().goodDaysToRobBank(*a) for a in [([1, 0, 1], 1), ([3, 3, 3, 3], 1), ([5, 4, 3, 2, 1], 1)]] == [[1], [1, 2], []]
# Bài 2
trace_runs([6, 5, 5, 4, 4, 5, 7, 7, 2], 2)
assert Solution().goodDaysToRobBank([6, 5, 5, 4, 4, 5, 7, 7, 2], 2) == good_days_brute([6, 5, 5, 4, 4, 5, 7, 7, 2], 2) == [3, 4]

# Bài 3
def nonincreasing_runs_ans(a):
    left = [0] * len(a)
    for i in range(1, len(a)):
        if a[i] <= a[i - 1]: left[i] = left[i - 1] + 1
    return left

# Bài 4
def max_time_ans(security):
    n = len(security); left = nonincreasing_runs_ans(security); right = [0] * n
    for i in range(n - 2, -1, -1):
        if security[i] <= security[i + 1]: right[i] = right[i + 1] + 1
    return max(min(l, r) for l, r in zip(left, right))

rng = random.Random(41)
for _ in range(500):
    sec = [rng.randint(0, 4) for _ in range(rng.randint(1, 15))]
    mt = max_time_ans(sec)
    assert good_days_brute(sec, mt) and not good_days_brute(sec, mt + 1)
print("Bài 4: max_time ví dụ 1 =", max_time_ans([5, 3, 3, 3, 5, 6, 2]))
print("Bài 1-4: OK")

Tính left (trái -> phải):
  i = 1: 5 <= 6? True -> left[1] = 1
  i = 2: 5 <= 5? True -> left[2] = 2
  i = 3: 4 <= 5? True -> left[3] = 3
  i = 4: 4 <= 4? True -> left[4] = 4
  i = 5: 5 <= 4? False -> left[5] = 0
  i = 6: 7 <= 5? False -> left[6] = 0
  i = 7: 7 <= 7? True -> left[7] = 1
  i = 8: 2 <= 7? True -> left[8] = 2
Tính right (phải -> trái):
  i = 7: 7 <= 2? False -> right[7] = 0
  i = 6: 7 <= 7? True -> right[6] = 1
  i = 5: 5 <= 7? True -> right[5] = 2
  i = 4: 4 <= 5? True -> right[4] = 3
  i = 3: 4 <= 4? True -> right[3] = 4
  i = 2: 5 <= 4? False -> right[2] = 0
  i = 1: 5 <= 5? True -> right[1] = 1
  i = 0: 6 <= 5? False -> right[0] = 0
  ngày 0: left = 0, right = 0 -> -
  ngày 1: left = 1, right = 1 -> -
  ngày 2: left = 2, right = 0 -> -
  ngày 3: left = 3, right = 4 -> TỐT
  ngày 4: left = 4, right = 3 -> TỐT
  ngày 5: left = 0, right = 2 -> -
  ngày 6: left = 0, right = 1 -> -
  ngày 7: left = 1, right = 0 -> -
  ngày 8: left = 2, right = 0 -> -
Bài 4: max_time ví dụ 1

In [22]:
# Bài 5 — dung sai c vi phạm mỗi phía
def good_days_tolerance(security, time, c):
    n = len(security)
    up = [0] + [1 if security[j] > security[j - 1] else 0 for j in range(1, n)]
    dn = [0] + [1 if security[j] < security[j - 1] else 0 for j in range(1, n)]
    PU = list(accumulate(up)); PD = list(accumulate(dn))
    return [i for i in range(time, n - time) if PU[i] - PU[i - time] <= c and PD[i + time] - PD[i] <= c]

def good_days_tolerance_brute(security, time, c):
    n = len(security)
    return [i for i in range(time, n - time)
            if sum(security[j] > security[j - 1] for j in range(i - time + 1, i + 1)) <= c
            and sum(security[j] > security[j + 1] for j in range(i, i + time)) <= c]

rng = random.Random(55)
for _ in range(1000):
    sec = [rng.randint(0, 4) for _ in range(rng.randint(1, 15))]; t = rng.randint(0, 6); c = rng.randint(0, 2)
    assert good_days_tolerance(sec, t, c) == good_days_tolerance_brute(sec, t, c)
    if c == 0: assert good_days_tolerance(sec, t, 0) == good_days_brute(sec, t)
print("Bài 5 ví dụ 1 với c = 0, 1:", good_days_tolerance([5, 3, 3, 3, 5, 6, 2], 2, 0), good_days_tolerance([5, 3, 3, 3, 5, 6, 2], 2, 1))

Bài 5 ví dụ 1 với c = 0, 1: [2, 3] [2, 3, 4]


## 20. Template nhớ nhanh + Checklist

```python
# Độ dài chuỗi đơn điệu liên tiếp từ hai phía
left = [0] * n; right = [0] * n
for i in range(1, n):
    if a[i] <= a[i - 1]: left[i] = left[i - 1] + 1      # không tăng, kết thúc ở i
for i in range(n - 2, -1, -1):
    if a[i] <= a[i + 1]: right[i] = right[i + 1] + 1    # không giảm, bắt đầu ở i
good = [i for i in range(n) if left[i] >= time and right[i] >= time]
```

**Checklist trước khi submit:**
- [ ] So sánh `≤` (cho phép bằng), không phải `<`.
- [ ] `right` tính **từ phải sang trái**.
- [ ] Thống nhất "số bước" (so với `time`) hay "số phần tử" (so với `time + 1`).
- [ ] Test `time = 0`, `2·time ≥ n`, mảng hằng.
- [ ] Prefix: bước trước là `(i − time, i]`, bước sau là `(i, i + time]`.